In [4]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')


Python  : 3.14.3
NumPy   : 2.4.2
Seed    : 414


In [5]:
# ── Cell 2.1: FastF1 Session Load ────────────────────────────────────────────
import fastf1
import os
import pandas as pd


# FastF1 cache — auto-create if missing
cache_path = os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)
print(f'FastF1 cache enabled: {cache_path}')
session = fastf1.get_session(2025, 'Australia', 'R')
session.load()

FastF1 cache enabled: /home/inzenfenix/Documents/data/fastf1_cache


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core    

In [21]:
# ── Cell 2.2: FastF1 Session Print ────────────────────────────────────────────

session_data = session.results
session_data['Position'] = pd.to_numeric(session_data['Position'], errors='coerce')

# Once we check that position is numeric, we get the first place and the amount of drivers for this race
drivers_amount = session_data.shape[0]
first_place = session.results.loc[session.results["Position"] == 1.0]
first_place_laps = first_place["Laps"][0]

print("Checking on 2025 Australia's Session")
print("Session Name:", session.name)
print("Shape: ", session.results.shape)
print("Event: ",session.event['EventName'])
print("Drivers: ", drivers_amount)
print("First place Laps: ", first_place_laps)

#Output of the the session metadata
print("\n----- Dataframe Metadata -----\n")
print("----- COLUMNS ----- \n", session.results.columns)
print("\n----- Lap data's head ----- \n", session.laps.head())

Checking on 2025 Australia's Session
Session Name: Race
Shape:  (20, 22)
Event:  Australian Grand Prix
Drivers:  20
First place Laps:  57.0

----- Dataframe Metadata -----

----- COLUMNS ----- 
 Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps'],
      dtype='object')

----- Lap data's head ----- 
                     Time Driver DriverNumber                LapTime  \
0 0 days 01:13:00.002000    VER            1 0 days 00:01:59.392000   
1 0 days 01:15:49.358000    VER            1                    NaT   
2 0 days 01:18:31.526000    VER            1                    NaT   
3 0 days 01:21:07.226000    VER            1                    NaT   
4 0 days 01:23:30.835000    VER            1 0 days 00:02:23.609000   

   LapNumber  Stint          

In [22]:
# ── Cell 3: Jolpica API Query ────────────────────────────────────────────
import requests


#Driver standings URL for 2025 formatted in JSON
api_url = "https://api.jolpi.ca/ergast/f1/2025/driverstandings/?format=json"

#We GET the request and check if we got a 200 status code, in which case we continue with the code
request = requests.get(api_url, timeout=60)
status_code = request.status_code
print("Status Code:", status_code)

if(status_code == 200):
    payload = request.json()
    #We get the standing data and then get the drivers list, the standings list only contains one object which is the positioning by the end of the season
    standings_data = payload["MRData"]["StandingsTable"]
    driver_list = standings_data["StandingsLists"][0]["DriverStandings"]
    print("Payload loaded succesfully!\n")

    print("Season", standings_data["season"])
    print("Round", standings_data["round"])

    #We flat the "Driver" json object for better readability and to format it as a Panda's DataFrame
    drivers_data = []

    for driver in driver_list:
        driver_dict = dict()

        driver_dict.update(driver["Driver"])

        driver_dict["position"] = driver["position"]
        driver_dict["points"] = driver["points"]
        driver_dict["wins"] = driver["wins"]

        drivers_data.append(driver_dict)

    df_drivers = pd.DataFrame(drivers_data)

    #We don't need the Wikipedia's url
    df_drivers.pop("url")
    
    print("Records:", len(drivers_data))
    print("Shape: ", df_drivers.shape)
    display(df_drivers.head(3)[["givenName","familyName", "nationality", "points"]])

Status Code: 200
Payload loaded succesfully!

Season 2025
Round 24
Records: 21
Shape:  (21, 10)


,givenName,familyName,nationality,points
0,Lando,Norris,British,423
1,Max,Verstappen,Dutch,421
2,Oscar,Piastri,Australian,410


## ── Cell 4: Data Shape Summary ────────────────────────────────────────────


### a) FastF1 Session - Australia

I chose 2025's Australia Race, mainly because of the results and the place.

> In 2025 Norris came first at the end of the session which was a change compared to 2024's winner Max Verstappen, for this reason comparing both seasons could be interesting to understand why this change happened.

> I used Australia because of its particular weather, Australia being as hot and dry as It is helps to benchmark cars, mainly their resistance to high intensity heat, and also thanks to Australia being so dry It can create an environment where the data has less variance which could definitely help once We have to analyze it.

### b) Shapes
FastF1 Session Shape: (20, 22)
Jolpica API' Driver Standings Shape: (21, 10)

### c) Observation
It was interesting to check how the JSON was structured, some data, mainly position, points and wins were outside of the Driver's data, which really helps to know what type and what importance does data has, the metadata was always on top and the deeper you went inside the JSON the more descriptive data became.